In [0]:

# ---- Install dependencies ----
%pip uninstall -y google google-cloud google-generativeai
%pip install faiss-cpu sentence-transformers google-generativeai
dbutils.library.restartPython()


In [0]:
# ---- Imports ----
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from pyspark.sql.functions import concat_ws

# ---- Load dataset ----
df = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/raw-data/banking/csv/Banking_Database.csv")
)

# ---- Convert rows to text ----
text_df = df.withColumn(
    "document",
    concat_ws(
        " | ",
        "Customer ID",
        "First Name",
        "Last Name",
        "Age",
        "City",
        "Account Type",
        "Account Balance",
        "Loan Type",
        "Loan Amount",
        "Loan Status",
        "Transaction Type",
        "Transaction Amount",
        "Anomaly"
    )
)

documents = [r.document for r in text_df.select("document").collect()]

# ---- Create embeddings ----
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(documents)

# ---- FAISS index ----
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

# ---- Retrieval function ----
def retrieve_context(question, top_k=2, max_chars=3000):
    q_emb = embedding_model.encode([question])
    _, idx = index.search(np.array(q_emb).astype("float32"), top_k)
    context = "\n".join([documents[i] for i in idx[0]])
    return context[:max_chars]

# ---- Gemini setup ----
genai.configure(api_key="Add your api key")

model = genai.GenerativeModel("gemini-2.5-flash")

# ---- Ask AI function ----
def ask_ai(question):
    context = retrieve_context(question)
    prompt = f"""
You are a banking analyst.
Answer strictly using the context.
If data is missing, say "Data not available".

Context:
{context}

Question:
{question}

Answer:
"""
    return model.generate_content(prompt).text

print("✅ System Ready: Data indexed + Gemini connected")


🏦 Customer & Account Insights

Which customers have the highest account balance?

How many customers are above 60 years of age?

Which account type is most commonly used?

List customers whose account balance is below ₹5,000.

Which cities have the maximum number of customers?

Identify customers with inactive accounts (no recent transactions).

💳 Transaction Analysis

What is the total transaction amount by transaction type?

Which customer has the highest single transaction?

List failed or suspicious transactions.

What percentage of transactions are credit vs debit?

Identify customers with frequent transactions in short time.

Show transactions marked as anomalies.

🚨 Anomaly & Risk Detection (Very Important)

How many records are flagged as anomalies?

Which customers have high transaction amount + anomaly flag?

Are anomalies more common in any specific account type?

Do anomalous transactions correlate with low account balance?

Which branch has the highest number of anomalies?

🏦 Loan Analysis

How many loans are approved vs rejected?

Which loan type has the highest average loan amount?

List customers with rejected loans.

What is the average interest rate per loan type?

Which customers have active loans with high outstanding balance?

What is the total loan exposure by branch?

💳 Credit Card Insights

Which customers have credit card balance close to credit limit?

What is the average credit limit by card type?

List customers who missed credit card payment due date.

Who has the maximum reward points?

Identify customers with high credit utilization ratio.

🧾 Customer Feedback & Service Quality

How many feedback cases are unresolved?

Which feedback type occurs most frequently?

What is the average resolution time for complaints?

Are unresolved cases linked to specific branches?

List customers with multiple complaints.

📊 Cross-Domain / Advanced Questions (Great for Demo)

Which customers have loans + credit cards + anomalies?

Identify high-value customers with low risk.

Are rejected loans more common for any age group?

Does transaction behavior impact loan approval?

Which customers require immediate risk review?

What patterns indicate potential fraud?

Summarize the financial health of customers by city.

In [0]:
# Run this once to create the widget at the top of your screen
dbutils.widgets.text("banking_query", "", "Ask your Banking Question:")

In [0]:
# --- RUN THIS CELL TO CHAT ---
user_query = dbutils.widgets.get("banking_query")

if user_query.strip() != "":
    # Note: Use the updated model name here if you haven't updated it globally
    model = genai.GenerativeModel("gemini-2.5-flash")
    
    # 1. Retrieve Context (from your existing function)
    context = retrieve_context(user_query)
    
    # 2. Ask the AI
    response = ask_ai(user_query)
    
    # 3. Display in a nice window
    displayHTML(f"""
        <div style="background-color: #f4f4f9; padding: 20px; border-radius: 10px; border-left: 5px solid #1a73e8;">
            <h4 style="color: #1a73e8; margin-top: 0;">Banking AI Response:</h4>
            <p style="font-size: 14px; color: #333;">{response}</p>
        </div>
    """)